In [11]:
import Pkg
Pkg.add("MLJFlux")

    Updating registry at `C:\Users\gianp\.julia\registries\General.toml`
   Resolving package versions...
   Installed PartialFunctions ─ v1.2.1
   Installed JLD2 ───────────── v0.5.15
   Installed Metalhead ──────── v0.9.5
   Installed MLJFlux ────────── v0.6.6
    Updating `C:\Users\gianp\.julia\environments\v1.11\Project.toml`
⌃ [094fc8d1] + MLJFlux v0.6.6
    Updating `C:\Users\gianp\.julia\environments\v1.11\Manifest.toml`
⌅ [033835bb] + JLD2 v0.5.15
⌃ [094fc8d1] + MLJFlux v0.6.6
  [dbeba491] + Metalhead v0.9.5
  [570af359] + PartialFunctions v1.2.1
        Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated -m`
Precompiling project...
   2171.9 ms  ✓ PartialFunctions
  27166.0 ms  ✓ JLD2
   2771.4 ms  ✓ JLD2 → UnPackExt
   7930.1 ms  ✓ Metalhead
   6873.8 ms  ✓ MLJFlux
  5 dependencies successfully precompiled in 48 seconds. 487

# Load required packages

In [5]:
using CSV, DataFrames
using MLJ
using MLJBase
using Statistics
using CSV
using DataFrames

# Load input data file

In [ ]:


# Define the path to the file
file_path = joinpath("input", "bank-additional-full.csv")

# Read the CSV into a DataFrame
df = CSV.read(file_path, DataFrame)

print("Data loaded successfully. Number of rows: $(nrow(df)), Number of columns: $(ncol(df))\n")
# Optional: preview the data
first(df, 5)

for col in names(df)
    col_type = eltype(df[!, col])
    num_missing = count(ismissing, df[!, col])
    println("Column: $(col) | Type: $(col_type) | Missing: $(num_missing)")
end


In [8]:
# Convert string columns to categorical
for col in names(df)
    if eltype(df[!, col]) <: AbstractString
        df[!, col] = categorical(df[!, col])
    end
end

# Define )
target = df.y
train = DataFrames.select(df, Not(:y))
first(train, 5)

Row,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
,Int64,Cat…,Cat…,Cat…,Cat…,Cat…,Cat…,Cat…,Cat…,Cat…,Int64,Int64,Int64,Int64,Cat…,Float64,Float64,Float64,Float64,Float64
1,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
2,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
3,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
4,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
5,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0


In [9]:
train, test = partition(eachindex(target), 0.8, shuffle=true)


([40128, 38532, 14517, 21880, 1365, 12249, 1083, 24048, 30705, 32977  …  1613, 7998, 17857, 15842, 8742, 26605, 3780, 18476, 15032, 8798], [11578, 20683, 9163, 16592, 11260, 35585, 32315, 37226, 674, 29749  …  553, 31857, 729, 4478, 1292, 38918, 3538, 19339, 16439, 23712])

In [12]:
# Logistic Regression
LR = @load LogisticClassifier pkg=MLJLinearModels

# Decision Tree
DT = @load DecisionTreeClassifier pkg=DecisionTree

# SVM
SVM = @load SVC pkg=LIBSVM

# Simple Neural Network (feed-forward)
NN = @load NeuralNetworkClassifier pkg=MLJFlux

# Another ANN (same as NN, but you can tune architecture)
ANN = NN


import MLJLinearModels ✔


┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\gianp\.julia\packages\MLJModels\ziReN\src\loading.jl:159


import MLJDecisionTreeInterface ✔
import MLJLIBSVMInterface ✔
import MLJFlux

┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\gianp\.julia\packages\MLJModels\ziReN\src\loading.jl:159
┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\gianp\.julia\packages\MLJModels\ziReN\src\loading.jl:159
┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\gianp\.julia\packages\MLJModels\ziReN\src\loading.jl:159


 ✔


MLJFlux.NeuralNetworkClassifier

In [15]:
models = Dict(
    "LR" => LR(),
    "DT" => DT(),
    "SVM" => SVM(),
    "NN" => NN(),
    "ANN" => ANN()
)

for (name, model) in models
    mach = machine(model, train, target)
    fit!(mach, rows=train)
    ŷ = predict(mach, rows=test)
    acc = accuracy(ŷ, y[test])
    println("Model: $(name) | Accuracy: $(round(acc, digits=4))")
end


┌ Warning: The number and/or types of data arguments do not match what the specified model
│ supports. Suppress this type check by specifying `scitype_check_level=0`.
│ 
│ Run `@doc LIBSVM.SVC` to learn more about your model's requirements.
│ 
│ Commonly, but non exclusively, supervised models are constructed using the syntax
│ `machine(model, X, y)` or `machine(model, X, y, w)` while most other models are
│ constructed with `machine(model, X)`.  Here `X` are features, `y` a target, and `w`
│ sample or class weights.
│ 
│ In general, data in `machine(model, data...)` is expected to satisfy
│ 
│     scitype(data) <: MLJ.fit_data_scitype(model)
│ 
│ In the present case:
│ 
│ scitype(data) = Tuple{AbstractVector{Count}, AbstractVector{Multiclass{2}}}
│ 
│ fit_data_scitype(model) = Union{Tuple{Table{<:AbstractVector{<:Continuous}}, AbstractVector{<:Finite}}, Tuple{Table{<:AbstractVector{<:Continuous}}, AbstractVector{<:Finite}, Any}}
└ @ MLJBase C:\Users\gianp\.julia\packages\MLJBase\7nGJF

DimensionMismatch: DimensionMismatch: Differing number of observations in input and target. 

In [14]:
using MLJBase: roc, confusion_matrix
for (name, model) in models
    mach = machine(model, X, target)
    fit!(mach, rows=train)
    ŷ = predict(mach, rows=test)
    roc_curve = roc(ŷ, target[test])
    cm = confusion_matrix(ŷ, target[test])
    println("Model: $(name)")
    println("ROC Curve: $(roc_curve)")
    println("Confusion Matrix:\n$(cm)")
end

UndefVarError: UndefVarError: `X` not defined in `Main`
Suggestion: check for spelling errors or missing imports.